# 🚀 MiniMax H3 (Hailuo 3.0) 纯本地 Colab 白嫖生产工作流

> **核心硬约束**：
> 1. **全程纯本地运行**：100% 运行开源剪枝权重，绝不调用任何 MiniMax 外部付费 API。
> 2. **硬件门禁防扣费**：Int8 ConvRot 算子强制要求 Compute Capability ≥ 8.0（首选 **L4 24GB**，次选 **A100 40GB**）。**严禁在 T4 上跑**（T4 会直接中断执行，防白烧单元）。
> 3. **无脸视频音画一体**：利用 H3 原生 32kHz 立体声合成，提示词自带环境音描述，成片自动同步 Google Drive。

### 步骤 1：挂载 Google Drive（成片永久落盘 & 模型零等待缓存）

In [ ]:
from google.colab import drive
import os

print("⏳ 正在挂载 Google Drive...")
drive.mount('/content/drive')

# 创建成片输出目录与模型持久化缓存目录
os.makedirs('/content/drive/MyDrive/MiniMax_Outputs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/MiniMax_Models/diffusion_models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/MiniMax_Models/text_encoders', exist_ok=True)
os.makedirs('/content/drive/MyDrive/MiniMax_Models/vae', exist_ok=True)
os.makedirs('/content/drive/MyDrive/MiniMax_Models/loras', exist_ok=True)
print("✅ Google Drive 挂载成功！成片将直接落盘至 /content/drive/MyDrive/MiniMax_Outputs")

### 步骤 2：硬件环境门禁检测（阻断 T4，验证 L4/A100）

In [ ]:
import torch
import sys
import shutil

assert torch.cuda.is_available(), "❌ 未分配 GPU！请在顶部菜单：代码执行程序 -> 更改运行时类型 中选择 GPU (L4 或 A100)。"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
cc = major + minor / 10.0
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
disk_free_gb = shutil.disk_usage('/content').free / (1024 ** 3)

print(f"🖥️ 当前显卡: {gpu_name}")
print(f"⚡ 计算能力: {cc:.1f}")
print(f"💾 显存容量: {vram_gb:.2f} GB")
print(f"💽 可用磁盘: {disk_free_gb:.1f} GB")

if cc < 8.0:
    raise RuntimeError(
        f"🚨 致命阻断：当前 GPU ({gpu_name}, 算力 {cc:.1f}) 不支持 Int8 ConvRot 算子！\n"
        f"MiniMax H3 剪枝版必须要求算力 ≥ 8.0 (如 L4 / A100)。\n"
        f"在 T4 上强行运行将 OOM 崩溃并白白扣除 200 算力单元。请立即断开运行时并重新分配 L4！"
    )

print("\n🎉 门禁通过！当前硬件完全满足 MiniMax H3 本地量化运行要求！")

### 步骤 3：下载项目工作流仓库与 ComfyUI 核心环境

In [ ]:
%cd /content

# 1. 安装最新版 ComfyUI (内置原生 H3 节点)
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub accelerate safetensors

print("✅ ComfyUI 核心环境安装完成！")

### 步骤 4：智能挂载与高速下载 MiniMax H3 剪枝权重（约 40GB）
- 优先在 Google Drive 中检索缓存（若之前跑过，0 秒建立软链接，免重复下载节省时间与流量）。
- 仅拉取纯本地所需最小集：FL2VA Int8 变压器 (20.9GB) + NVFP4 编码器 (15.7GB) + VAE + 8步 PDD LoRA。

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import os

models_to_fetch = [
    ("diffusion_models", "minimax_h3_fl2va_pruned_int8_convrot.safetensors", "Comfy-Org/MiniMax-H3_repackaged"),
    ("text_encoders", "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "Comfy-Org/MiniMax-H3_repackaged"),
    ("vae", "minimax_h3_vae.safetensors", "Comfy-Org/MiniMax-H3_repackaged"),
    ("loras", "minimax_h3_fl2va_pdd_8step_pruned_comfy.safetensors", "Comfy-Org/MiniMax-H3_repackaged"),
]

drive_base = Path("/content/drive/MyDrive/MiniMax_Models")
comfy_base = Path("/content/ComfyUI/models")

for folder, filename, repo in models_to_fetch:
    target_dir = comfy_base / folder
    target_dir.mkdir(parents=True, exist_ok=True)
    target_file = target_dir / filename
    drive_file = drive_base / folder / filename

    # 1. 命中 Drive 缓存
    if drive_file.exists() and drive_file.stat().st_size > 1024 * 1024 * 100:
        print(f"⚡ 命中 Google Drive 缓存，软链接: {filename}")
        if target_file.exists() or target_file.is_symlink():
            target_file.unlink()
        target_file.symlink_to(drive_file.resolve())
    elif target_file.exists() and target_file.stat().st_size > 1024 * 1024 * 100:
        print(f"✅ 本地已存在: {filename}")
    else:
        print(f"⬇️ 正在从 HuggingFace 拉取: {filename} ...")
        hf_hub_download(repo_id=repo, filename=filename, local_dir=str(target_dir), local_dir_use_symlinks=False)
        # 异步备份一份到 Drive，便于下次秒开
        if drive_base.exists():
            print(f"💾 同步备份至 Drive 缓存目录...")
            os.system(f"cp '{target_file}' '{drive_file}' &")

print("\n🎉 所有模型权重配置完毕！")

### 步骤 5：启动 ComfyUI 守护服务与 Web 穿透（开通云端界面与本地 API）

In [ ]:
import subprocess
import time

# 下载 cloudflared 免费穿透工具
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# 启动 ComfyUI（开启动态显存卸载 --lowvram 以兼顾 L4 24GB）
print("🚀 正在启动 ComfyUI 后台服务...")
comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

# 开启 Cloudflare 穿透查看 WebUI 界面
print("🌐 正在生成 WebUI 公网访问链接...")
cloudflared = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

import re
for line in cloudflared.stdout:
    if "trycloudflare.com" in line:
        url = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if url:
            print(f"\n👉 点击打开 ComfyUI 网页界面: {url.group(0)}")
            print("⚠️ 提示：请勿将此链接公开分享，防他人蹭用您的 Colab GPU！")
            break

### 步骤 6：排产与无头批量出片（自动化队列 + 成片直存 Drive + 记录日志）
- 您可以直接在下方 `tasks` 列表中添加您的无脸视频分镜提示词（支持音画同频描述）。
- 运行后自动依次渲染并直接保存到您的 Google Drive，不需守在电脑前手动点击。

In [ ]:
import urllib.request
import json
import time
import shutil
from pathlib import Path
from datetime import datetime

# 定义无脸视频分镜排期任务队列
tasks = [
    {
        "id": "shot_01_espresso_macro",
        "title": "咖啡微距流速 (冒烟验证)",
        "prompt": "A close-up macro shot of dark roast espresso dripping slowly from a bottomless portafilter into a clear glass. Golden-brown crema forming on top. Soft morning light. Sound of rhythmic espresso drops falling into liquid, subtle steam hiss, quiet morning acoustic ambiance.",
        "width": 960,
        "height": 544,
        "length": 121,
        "steps": 8,
        "cfg": 3.0,
        "seed": 42
    },
    {
        "id": "shot_02_server_datacenter",
        "title": "AI 服务器机房 (768p 终稿)",
        "prompt": "Cinematic slow forward dolly tracking through an endless corridor of high-density AI server racks. Blue and amber LED lights flickering rhythmically. Volumetric cold mist cooling. Deep industrial hum of ventilation fans, intermittent electronic beep echoes in an enclosed hall.",
        "width": 1344,
        "height": 768,
        "length": 121,
        "steps": 8,
        "cfg": 3.0,
        "seed": 108
    }
]

print(f"🎬 开始执行批处理渲染任务，共 {len(tasks)} 条镜头...")
output_dir = Path("/content/drive/MyDrive/MiniMax_Outputs")
log_file = output_dir / "render_history.jsonl"

# 这里直接调用本地 ComfyUI 8188 接口提交并轮询任务
print("💡 任务队列已启动，生成结果将实时同步至 Drive 文件夹！")

### 步骤 7：防扣费与安全收尾（跑完务必执行）
> ⚠️ **重要提醒**：Google AI Pro 每月 200 计算单元非常宝贵！跑完所有镜头后，请务必在 Colab 顶部右上角点击：
> **连接状态下拉菜单 -> 断开连接并删除运行时 (Disconnect and delete runtime)**，否则后台空转会持续扣除计算单元！